# FourCastNet Registry GPU Smoke (Studio/Notebook)

Este notebook valida FourCastNet desde **Model Registry + S3 artifacts** en GPU de SageMaker Studio/Notebook.

- No crea endpoint
- No aprueba model package
- No usa Processing, Batch Transform o Training


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

MODEL_PACKAGE_ARN = "arn:aws:sagemaker:us-east-1:725644097028:model-package/sbnai-fourcastnet-fcn-v0/1"
INPUT_TENSOR_S3_URI = "s3://chucaw-data-platinum-processed-725644097028-us-east-1-an/sagemaker/fourcastnet/fcn-v0/input/input_tensor.npy"
REGION = "us-east-1"
PROFILE = ""  # dejar vacio en Studio; usar perfil local si aplica

repo_root = Path.cwd().resolve()
if not (repo_root / "src/fourcastnet/notebook_forward_smoke.py").exists() and (repo_root.parent / "src/fourcastnet/notebook_forward_smoke.py").exists():
    repo_root = repo_root.parent

script_path = repo_root / "src/fourcastnet/notebook_forward_smoke.py"
assert script_path.exists(), f"No se encontro script: {script_path}"

report_dir = repo_root / "artifacts/fourcastnet/notebook_smoke"
report_dir.mkdir(parents=True, exist_ok=True)
metadata_report = report_dir / "notebook_metadata_only_report.json"
forward_report = report_dir / "notebook_forward_report.json"

print("Repo root:", repo_root)
print("Script:", script_path)


In [ ]:
cmd = [
    sys.executable,
    str(script_path),
    "--model-package-arn", MODEL_PACKAGE_ARN,
    "--input-tensor-s3-uri", INPUT_TENSOR_S3_URI,
    "--mode", "metadata_only",
    "--region", REGION,
    "--output-report", str(metadata_report),
]
if PROFILE:
    cmd.extend(["--profile", PROFILE])

print("Running metadata_only...")
subprocess.run(cmd, check=False)

metadata = json.loads(metadata_report.read_text(encoding="utf-8"))
print(json.dumps({
    "mode": metadata.get("mode"),
    "ok": metadata.get("ok"),
    "result": metadata.get("result"),
    "model_package_status": metadata.get("model_package_status"),
    "model_approval_status": metadata.get("model_approval_status"),
    "fourcastnet_proven": metadata.get("fourcastnet_proven"),
}, indent=2))


In [ ]:
cmd = [
    sys.executable,
    str(script_path),
    "--model-package-arn", MODEL_PACKAGE_ARN,
    "--input-tensor-s3-uri", INPUT_TENSOR_S3_URI,
    "--mode", "forward",
    "--max-runtime-guard",
    "--region", REGION,
    "--output-report", str(forward_report),
]
if PROFILE:
    cmd.extend(["--profile", PROFILE])

print("Running forward...")
subprocess.run(cmd, check=False)

forward = json.loads(forward_report.read_text(encoding="utf-8"))
print(json.dumps({
    "mode": forward.get("mode"),
    "ok": forward.get("ok"),
    "result": forward.get("result"),
    "model_package_status": forward.get("model_package_status"),
    "model_approval_status": forward.get("model_approval_status"),
    "fourcastnet_proven": forward.get("fourcastnet_proven"),
    "forward": forward.get("forward", {}),
}, indent=2))


Criterio de prueba:
- `metadata_only` exitoso valida acceso a Model Registry + S3 artifacts.
- `forward` exitoso con `fourcastnet_proven=true` es la evidencia requerida de modelo probado en GPU.
